# 16 — Stress Scenario Design and Calibration

## Designing a transparent stress scenario

The completed base capital calculation is the starting point. This notebook identifies every shock and separates synthetic project sensitivities from Basel requirements.

Stress testing asks how losses, RWA and capital could change under severe but plausible conditions. Basel requires a meaningful and reasonably conservative test but does not prescribe the numerical multipliers used here.

### The scenario terms used below

| Term | Simple meaning |
|---|---|
| Macroeconomic path | a time path for variables such as GDP, unemployment, rates and property prices |
| Satellite model | a model linking macroeconomic variables to portfolio risk drivers |
| Multiplier | a proportional shock applied to a base value |
| Add-on | an absolute increase added to a base value |

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
from basel_credit_risk.config import load_all_config
from basel_credit_risk.notebook_support import display, ensure_outputs

CRORE = 10_000_000
COLORS = ["#0B3A53", "#1F77B4", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"]
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
configs = load_all_config(ROOT / "config")
ensure_outputs(ROOT)
loans = pd.read_csv(ROOT / "data/processed/current_calculated.csv.gz", low_memory=False)
print(f"Portfolio represented: {len(loans):,} synthetic exposures")

Portfolio represented: 30,000 synthetic exposures


## Synthetic sensitivities used here

Every multiplier, add-on, collateral shock, utilisation shock and loss rate below is synthetic. None is a Basel-prescribed adverse or severe value.

In [2]:
scenario_assumptions = pd.DataFrame(configs["stress_scenarios"]["scenarios"]).T
display(scenario_assumptions.style.format({
    "pd_multiplier": "{:.2f}x", "cyclical_sector_pd_multiplier": "{:.2f}x", "retail_pd_multiplier": "{:.2f}x",
    "lgd_addon": "{:.1%}", "collateral_shock": "{:.1%}", "undrawn_utilisation_addon": "{:.1%}", "loss_rate": "{:.1%}",
}))

,pd_multiplier,cyclical_sector_pd_multiplier,retail_pd_multiplier,lgd_addon,collateral_shock,undrawn_utilisation_addon,loss_rate
base,1.00x,1.00x,1.00x,0.0%,0.0%,0.0%,0.0%
adverse,1.35x,1.65x,1.50x,7.5%,12.0%,10.0%,1.2%
severe,1.85x,2.80x,2.35x,15.0%,25.0%,25.0%,3.0%


The general PD multiplier applies first. Higher cyclical-sector or retail multipliers replace it for the named portfolios. LGD and utilisation changes are add-ons, while the collateral shock reduces property and collateral values.

## How a bank would calibrate the shocks

A bank would begin with approved macroeconomic paths and estimate how those paths change default, recovery, utilisation, revenue, expenses and capital over the stress horizon. Historical data provide the first estimate; expert overlays are documented when the history does not contain a comparable event.

In [3]:
calibration = pd.DataFrame([
    ["PD", "GDP, unemployment, interest rates, sector output, borrower grade history", "Estimate grade or segment default models conditional on the scenario path"],
    ["LGD", "Defaulted-loan recoveries, collateral prices, recovery time and costs", "Link recoveries and collateral haircuts to downturn conditions"],
    ["EAD / utilisation", "Monthly limits, drawings, cancellations and defaults", "Estimate additional drawings before default by product and borrower condition"],
    ["Capital", "Credit losses, market losses, operational losses, income, expenses, taxes and distributions", "Project the capital numerator and RWA denominator across the horizon"],
], columns=["Risk driver", "Typical data", "Calibration route"])
display(calibration)

,Risk driver,Typical data,Calibration route
0,PD,"GDP, unemployment, interest rates, sector output, borrower grade history",Estimate grade or segment default models conditional on the scenario path
1,LGD,"Defaulted-loan recoveries, collateral prices, recovery time and costs",Link recoveries and collateral haircuts to downturn conditions
2,EAD / utilisation,"Monthly limits, drawings, cancellations and defaults",Estimate additional drawings before default by product and borrower condition
3,Capital,"Credit losses, market losses, operational losses, income, expenses, taxes and distributions",Project the capital numerator and RWA denominator across the horizon


The project deliberately uses transparent sensitivities because the required historical macroeconomic and bank outcome data are not present. The next notebook applies these assumptions to individual loans and shows the before-and-after changes.